In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

[Top 10 Vehicles in Malaysia](https://data.gov.my/dashboard/car-popularity)

[Guide to calculating insurance premium](https://bengkelbergerak.my/en/blog/kira-insurans-kereta)

[Premium calculator by Carso](https://www.carso.my/tool/car-insurance-calculator)

[Flooding risk weightage](https://www.dosm.gov.my/uploads/content-downloads/file_20220929154540.pdf)

In [2]:
# ============================================================================
# INITIAL COHORT GENERATION - single function, all assumptions in CohortConfig
# ============================================================================

ENGINE_CAPACITY_BANDS = [
    "0 to 1,400 cc / EV up to 70 kW",
    "1,401 to 1,650 cc / EV 71 - 100 kW",
    "1,651 - 2,200 cc / EV 101 - 125 kW",
    "2,201 - 3,050 cc / EV 126 - 150 kW",
    "3,051 - 4,100 cc / EV 151 - 200 kW",
    "4,101 - 4,250 cc / EV 201 - 250 kW",
    "4,251 - 4,400 cc / EV 251 - 300 kW",
    "Over 4,400 cc / EV > 300 kW",
]


@dataclass
class CohortConfig:
    """All settings/assumptions for initial cohort generation."""

    n: int = 100000
    coverage_pct: dict = field(default_factory=lambda: {
        'Comprehensive': 0.65, 'TPFT': 0.20, 'TPO': 0.15})
    vehicle_pct: dict = field(default_factory=lambda: {'ICE': 0.90, 'EV': 0.10})
    sa_stats: dict = field(default_factory=lambda: {
        'ICE': {'lambda': 50000, 'spread': 0.5},
        'EV': {'lambda': 80000, 'spread': 0.5}})
    region_pct: dict = field(default_factory=lambda: {
        'Peninsular Malaysia': 0.80,
        'East Malaysia (Sabah, Sawarak & Labuan)': 0.20})
    generation_pct: dict = field(default_factory=lambda: {
        'Gen-Z': 0.40, 'Millennial': 0.40, 'Boomers': 0.15, 'Senior': 0.05})
    age_bands: dict = field(default_factory=lambda: {
        'Gen-Z': (18, 28), 'Millennial': (28, 46),
        'Boomers': (46, 66), 'Senior': (66, 76)})
    gender_pct: dict = field(default_factory=lambda: {'Male': 0.55, 'Female': 0.45})
    marital_pct: dict = field(default_factory=lambda: {
        'Gen-Z': {'Single': 0.85, 'Married': 0.15},
        'Millennial': {'Single': 0.50, 'Married': 0.50},
        'Boomers': {'Single': 0.20, 'Married': 0.80},
        'Senior': {'Single': 0.20, 'Married': 0.80}})
    car_age_median: dict = field(default_factory=lambda: {
        'Gen-Z': 2.0, 'Millennial': 3.5, 'Boomers': 5.0, 'Senior': 5.5})
    car_age_sigma: float = 1.5
    risk_pct: dict = field(default_factory=lambda: {
        'Peninsular Malaysia': {
            'FLOOD_RISK': [0.60, 0.40], 'THEFT_RISK': [0.40, 0.60]},
        'East Malaysia (Sabah, Sawarak & Labuan)': {
            'FLOOD_RISK': [0.00, 1.00], 'THEFT_RISK': [0.15, 0.85]}})
    ncd_table: dict = field(default_factory=lambda: {
        0: 0.00, 1: 0.25, 2: 0.30, 3: 0.3833, 4: 0.45, 5: 0.55})
    ncd_entry_years: list = field(default_factory=lambda: [0, 1, 2, 3, 4, 5])
    ncd_entry_weights: list = field(default_factory=lambda: [0.30, 0.22, 0.16, 0.13, 0.10, 0.09])
    cohort_year: int = 2026
    engine_weights: list = field(default_factory=lambda: [0.25, 0.20, 0.18, 0.15, 0.10, 0.07, 0.03, 0.02])
    seed: int = 42


def _p(weights):
    """Normalise weights to sum exactly to 1 (float-safe for np.random.choice)."""
    a = np.array(list(weights), dtype=float)
    return a / a.sum()


def generate_dataset(cfg, seed=None, n=None, cohort_year=None, polid_prefix='INIT'):
    """Generate a complete policy book from CohortConfig assumptions.

    Args:
        cfg: CohortConfig with all settings/assumptions
        seed: RNG seed (defaults to cfg.seed)
        n: number of policies (defaults to cfg.n)
        cohort_year: policy base year (defaults to cfg.cohort_year)
        polid_prefix: 'INIT' for the initial book, 'ENT' for new entrants

    Returns:
        pd.DataFrame with all policy attributes + premium columns
    """
    n = int(n if n is not None else cfg.n)
    rng = np.random.default_rng(int(seed if seed is not None else cfg.seed))
    base_year = int(cohort_year if cohort_year is not None else cfg.cohort_year)

    df = pd.DataFrame(index=range(n))

    # Coverage / vehicle / region
    df['COVERAGE_TYPE'] = rng.choice(list(cfg.coverage_pct), size=n, p=_p(cfg.coverage_pct.values()))
    df['VEHICLE_TYPE'] = rng.choice(list(cfg.vehicle_pct), size=n, p=_p(cfg.vehicle_pct.values()))
    df['REGION'] = rng.choice(list(cfg.region_pct), size=n, p=_p(cfg.region_pct.values()))

    # Sum assured: log-normal per vehicle type, rounded to RM 1,000
    sa = np.zeros(n)
    for vt, stats in cfg.sa_stats.items():
        m = df['VEHICLE_TYPE'].values == vt
        sa[m] = rng.lognormal(mean=np.log(stats['lambda']), sigma=stats['spread'],
                              size=int(m.sum()))
    df['SUM_ASSURED'] = np.round(sa / 1000) * 1000

    # Engine capacity: fixed hand-set mix (small cars dominant)
    df['ENGINE_CAPACITY'] = rng.choice(ENGINE_CAPACITY_BANDS, size=n,
                                       p=_p(cfg.engine_weights))

    # Driver profile: category by weights, age drawn from category band
    df['DRIVER_AGE_CAT'] = rng.choice(list(cfg.generation_pct), size=n,
                                      p=_p(cfg.generation_pct.values()))
    cats = df['DRIVER_AGE_CAT'].values
    lo = np.array([cfg.age_bands[c][0] for c in cats])
    hi = np.array([cfg.age_bands[c][1] for c in cats])
    df['DRIVER_AGE'] = rng.integers(lo, hi, size=n)
    df['DRIVER_GENDER'] = rng.choice(list(cfg.gender_pct), size=n,
                                     p=_p(cfg.gender_pct.values()))
    marital = np.empty(n, dtype=object)
    for cat, probs in cfg.marital_pct.items():
        m = cats == cat
        marital[m] = rng.choice(list(probs), size=int(m.sum()), p=_p(probs.values()))
    df['MARITAL_STATUS'] = marital

    # Vehicle age at inception, capped 0-10
    car_age = np.zeros(n)
    for cat, med in cfg.car_age_median.items():
        m = cats == cat
        car_age[m] = np.clip(np.round(med + rng.normal(0, cfg.car_age_sigma,
                                                       size=int(m.sum()))), 0, 10)
    df['CAR_AGE'] = car_age.astype(int)

    # Region flood/theft risk flags
    for col in ('FLOOD_RISK', 'THEFT_RISK'):
        out = np.zeros(n, dtype=bool)
        for region, probs in cfg.risk_pct.items():
            m = df['REGION'].values == region
            out[m] = rng.choice([True, False], size=int(m.sum()), p=_p(probs[col]))
        df[col] = out

    # NCD entry mix
    df['NCD_YEARS'] = rng.choice(cfg.ncd_entry_years, size=n, p=_p(cfg.ncd_entry_weights))
    df['NCD_LEVEL'] = df['NCD_YEARS'].apply(lambda y: cfg.ncd_table.get(int(min(y, 5)), 0.55))
    df['COHORT_YEAR'] = base_year

    # POLID: deterministic prefix-year-sequence (INIT/ENT)
    df['POLID'] = [f"{polid_prefix}{base_year}-{i + 1:06d}" for i in range(n)]

    # Premium: BASIC from tariff, TOTAL_LOADING, FINAL with SST
    df['BASIC_PREMIUM'] = df.apply(calculate_premium, axis=1)
    df['TOTAL_LOADING'] = df.apply(
        lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
    )
    df['FINAL_PREMIUM_SST'] = df.apply(compute_final_premium, axis=1)

    # Apply the canonical schema dtypes
    for col, dt in dtype_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dt)
    return df

In [3]:
features = [
    "POLID",

    # Plan details (required to obtain basic premium)
    "COVERAGE_TYPE",
    "VEHICLE_TYPE", # <- Not really needed for calculating premium, added for sum assured
    "CAR_AGE", # <- Not really needed for calculating premium, added for loadings as time progress
    "SUM_ASSURED",
    "REGION",
    "ENGINE_CAPACITY",

    # Insured details
    "DRIVER_AGE_CAT",
    "DRIVER_AGE",
    "DRIVER_GENDER",
    "MARITAL_STATUS",

    # Premium refinements
    "FLOOD_RISK",
    "THEFT_RISK",
    

    # Output from plan details
    "BASIC_PREMIUM",
    "FINAL_PREMIUM_SST",

    # Calculation specific variables
    "NCD_LEVEL",
    "NCD_YEARS",
    "COHORT_YEAR"
]

# Define dtypes for each column
dtype_dict = {
    "POLID": "string",

    # Plan details (required to obtain basic premium)
    "COVERAGE_TYPE": "category",
    "VEHICLE_TYPE": "category",
    "CAR_AGE": "int64",
    "SUM_ASSURED": "float64",
    "REGION": "category",
    "ENGINE_CAPACITY": "category",

    # Insured details
    "DRIVER_AGE_CAT": "category",
    "DRIVER_AGE": "int64",
    "DRIVER_GENDER": "category",
    "MARITAL_STATUS": "category",

    # Refinements
    "FLOOD_RISK": "boolean",
    "THEFT_RISK": "boolean",

    # Output from plan details
    "BASIC_PREMIUM": "float64",
    "FINAL_PREMIUM_SST": "float64",

    # Calculation specific variables
    "NCD_LEVEL": "float64",
    "NCD_YEARS": "int64",
    "COHORT_YEAR": "int64"
}

num_dataset = 100000
print(f"num_dataset = {num_dataset}")

num_dataset = 100000


In [4]:
# Coverage / vehicle / sum-assured / region / engine-capacity generation
# moved into generate_dataset() (cell 050b235c).

In [5]:
# Driver age category / age / gender / marital-status generation
# moved into generate_dataset() (cell 050b235c).

In [6]:
# CAR_AGE generation moved into generate_dataset() (cell 050b235c).
# age_to_cat removed: DRIVER_AGE_CAT is frozen at inception.

In [7]:
# Region flood/theft risk assignment moved into generate_dataset() (cell 050b235c).

In [8]:
# NCD setup - exact PIAM table progression (single source: CohortConfig)
COHORT_CONFIG = CohortConfig()

COHORT_YEAR = COHORT_CONFIG.cohort_year
NCD_TABLE = COHORT_CONFIG.ncd_table
NCD_ENTRY_YEARS = COHORT_CONFIG.ncd_entry_years
NCD_ENTRY_WEIGHTS = COHORT_CONFIG.ncd_entry_weights

print('NCD setup ready (from CohortConfig). Entry NCD mix:')
print(pd.Series(NCD_ENTRY_WEIGHTS, index=NCD_ENTRY_YEARS)
      .map(lambda x: f'{x:.1%}').to_string())

NCD setup ready (from CohortConfig). Entry NCD mix:
0    30.0%
1    22.0%
2    16.0%
3    13.0%
4    10.0%
5     9.0%


In [9]:
# ============================================================================
# BASIC PREMIUM - Schedule of Motor Tariff 2015 (Form 5 Mathematics Ch.3)
# Graduated tariff:
#   Comprehensive = first-RM1,000 rate + PER_EXTRA x ceil((SA-1000)/1000)
#   TPFT (Third Party, Fire & Theft) = 0.75 x Comprehensive basic (Example 3)
#   TPO = flat tariff rate (no sum-assured scaling)
# ============================================================================

PER_EXTRA = {
    'Peninsular Malaysia': 26.00,
    'East Malaysia (Sabah, Sawarak & Labuan)': 20.30,
}

MOTOR_TARIFF = {
    'Peninsular Malaysia': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 273.80,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 305.50,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 339.10,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 372.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 404.30,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 436.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 469.60,
            'Over 4,400 cc / EV > 300 kW': 501.30,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 120.60,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 135.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 151.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 167.40,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 181.80,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 196.20,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 212.40,
            'Over 4,400 cc / EV > 300 kW': 226.80,
        },
    },
    'East Malaysia (Sabah, Sawarak & Labuan)': {
        'Comprehensive': {
            '0 to 1,400 cc / EV up to 70 kW': 196.20,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 220.00,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 243.90,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 266.50,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 290.40,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 313.00,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 336.90,
            'Over 4,400 cc / EV > 300 kW': 359.50,
        },
        'TPO': {
            '0 to 1,400 cc / EV up to 70 kW': 67.50,
            '1,401 to 1,650 cc / EV 71 - 100 kW': 75.60,
            '1,651 - 2,200 cc / EV 101 - 125 kW': 85.20,
            '2,201 - 3,050 cc / EV 126 - 150 kW': 93.60,
            '3,051 - 4,100 cc / EV 151 - 200 kW': 101.70,
            '4,101 - 4,250 cc / EV 201 - 250 kW': 110.10,
            '4,251 - 4,400 cc / EV 251 - 300 kW': 118.20,
            'Over 4,400 cc / EV > 300 kW': 126.60,
        },
    },
}


def comprehensive_basic(row):
    first = MOTOR_TARIFF[row['REGION']]['Comprehensive'][row['ENGINE_CAPACITY']]
    units = int(np.ceil(max(0.0, (row['SUM_ASSURED'] - 1000.0) / 1000.0)))
    return first + PER_EXTRA[row['REGION']] * units


def calculate_premium(row):
    """Basic premium per Schedule of Motor Tariff 2015 (graduated)."""
    coverage = row['COVERAGE_TYPE']
    if coverage == 'Comprehensive':
        basic = comprehensive_basic(row)
    elif coverage == 'TPFT':
        basic = round(0.75 * comprehensive_basic(row) + 1e-9, 2)
    else:  # TPO - flat tariff rate
        basic = MOTOR_TARIFF[row['REGION']]['TPO'][row['ENGINE_CAPACITY']]
    return round(float(basic), 2)


# Textbook cross-check (Form 5 Example 3: Peninsular, 1,650cc band, SA 60,000)
def _mk(cov):
    return pd.Series({'COVERAGE_TYPE': cov, 'REGION': 'Peninsular Malaysia',
                      'ENGINE_CAPACITY': '1,401 to 1,650 cc / EV 71 - 100 kW',
                      'SUM_ASSURED': 60000.0})


assert abs(calculate_premium(_mk('Comprehensive')) - 1839.50) < 0.01
assert abs(calculate_premium(_mk('TPFT')) - 1379.63) < 0.01
assert abs(calculate_premium(_mk('TPO')) - 135.00) < 0.01
print('Tariff check OK (Form 5 Ex.3): Comp RM1,839.50 | TPFT RM1,379.63 | TPO RM135.00')

# BASIC_PREMIUM applied inside generate_dataset (cell 050b235c)

Tariff check OK (Form 5 Ex.3): Comp RM1,839.50 | TPFT RM1,379.63 | TPO RM135.00


In [10]:

# Rating loadings: driver age category + vehicle age
# Driver loading: Gen-Z highest (inexperience); experienced cohorts lower.
# Car loading: increases linearly with vehicle age (older car = higher risk).

DRIVER_AGE_LOADING = {
    "Gen-Z": 1.20,
    "Millennial": 1.05,
    "Boomers": 1.00,
    "Senior": 1.05,
}

def driver_age_loading(age_cat):
    return DRIVER_AGE_LOADING.get(age_cat, 1.00)

def car_age_loading(car_age):
    """Linear vehicle-age loading; car age capped at 10 years."""
    return 1 + 0.03 * min(int(car_age), 10)

def total_loading(age_cat, car_age):
    """Combined driver x vehicle loading applied to the premium."""
    return driver_age_loading(age_cat) * car_age_loading(car_age)


In [11]:

# FINAL_PREMIUM_SST: BASIC x driver/car loading x (1 - NCD) x 1.1^risk flags + 8% SST
# NCD discount applies to ALL coverages (both Comprehensive and TPO).
SST_RATE = 0.08  # Changeable variable - current SST rate in Malaysia

def compute_final_premium(row, sst_rate=SST_RATE):
    """Compute final premium with rating loadings, NCD discount, risk multipliers, SST."""
    loading = total_loading(row['DRIVER_AGE_CAT'], row['CAR_AGE'])
    ncd_discount = 1 - row.get('NCD_LEVEL', 0.0)
    risk_multiplier = 1.1 ** (int(row['FLOOD_RISK']) + int(row['THEFT_RISK']))
    final = row['BASIC_PREMIUM'] * loading * ncd_discount * risk_multiplier * (1 + sst_rate)
    return round(float(final), 2)

# FINAL_PREMIUM_SST / TOTAL_LOADING applied inside generate_dataset (cell 050b235c)


In [12]:
# Build the initial cohort (single generator call)
df = generate_dataset(COHORT_CONFIG, seed=COHORT_CONFIG.seed)

print(f"Initial cohort: {len(df):,} policies, year {COHORT_CONFIG.cohort_year}")
print(df[['POLID', 'COVERAGE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE', 'CAR_AGE',
          'NCD_LEVEL', 'BASIC_PREMIUM', 'FINAL_PREMIUM_SST']].head(5)
      .to_string(index=False))
print(df.dtypes)

Initial cohort: 100,000 policies, year 2026
          POLID COVERAGE_TYPE DRIVER_AGE_CAT  DRIVER_AGE  CAR_AGE  NCD_LEVEL  BASIC_PREMIUM  FINAL_PREMIUM_SST
INIT2026-000001          TPFT     Millennial          34        4        0.0        1170.83            1635.75
INIT2026-000002 Comprehensive     Millennial          44        2        0.0        3284.60            3948.22
INIT2026-000003           TPO        Boomers          53        4        0.0          93.60             113.22
INIT2026-000004          TPFT     Millennial          34        2        0.3        1438.13            1210.08
INIT2026-000005 Comprehensive          Gen-Z          18        2        0.3        1516.60            1458.41
COVERAGE_TYPE        category
VEHICLE_TYPE         category
REGION               category
SUM_ASSURED           float64
ENGINE_CAPACITY      category
DRIVER_AGE_CAT       category
DRIVER_AGE              int64
DRIVER_GENDER        category
MARITAL_STATUS       category
CAR_AGE             

In [13]:

# Claim Frequency Model (Poisson GLM, log-linear)
# lambda = exp(log_lambda) * coverage_multiplier
# Base exp(-2.00) ~ 0.135 claims/year - realistic Malaysian market level
# (tweak for realistic LR)

CLAIM_FREQUENCY_BASE = -2.00

def compute_claim_lambda(row):
    """Compute Poisson rate lambda via log-linear rating model."""
    log_lambda = CLAIM_FREQUENCY_BASE

    cat = row['DRIVER_AGE_CAT']
    if cat == 'Gen-Z':
        log_lambda += 0.40        # young drivers: higher risk
    elif cat == 'Senior':
        log_lambda += 0.26        # seniors: moderate increase

    # Young male interaction
    if cat == 'Gen-Z' and row['DRIVER_GENDER'] == 'Male':
        log_lambda += 0.05

    # EV proxy (higher power/repair exposure)
    if row['VEHICLE_TYPE'] == 'EV':
        log_lambda += 0.05

    # Risk flags
    if row['FLOOD_RISK']:
        log_lambda += 0.20
    if row['THEFT_RISK']:
        log_lambda += 0.10

    # Vehicle age: older cars carry higher breakdown/repair frequency
    log_lambda += 0.03 * row['CAR_AGE']

    # NCD safety credit: claim-free drivers are safer
    log_lambda -= 0.05 * row['NCD_YEARS']

    freq = np.exp(log_lambda)

    # Coverage multiplier: TPO has no own-damage exposure
    # Coverage multiplier: TPO no own-damage; TPFT fire/theft only (0.60)
    mult = 0.45 if row['COVERAGE_TYPE'] == 'TPO' else (0.60 if row['COVERAGE_TYPE'] == 'TPFT' else 1.00)
    return freq * mult


df['CLAIM_LAMBDA'] = df.apply(compute_claim_lambda, axis=1)

print('Claim frequency model (Poisson GLM) applied')
print(f"Mean lambda: {df['CLAIM_LAMBDA'].mean():.4f}")
print(f"Min lambda: {df['CLAIM_LAMBDA'].min():.4f}, "
      f"Max lambda: {df['CLAIM_LAMBDA'].max():.4f}")
print(f"TPO mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPO', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"Comp mean lambda: {df.loc[df['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_LAMBDA'].mean():.4f}")
print(f"TPFT mean lambda: {df.loc[df['COVERAGE_TYPE']=='TPFT', 'CLAIM_LAMBDA'].mean():.4f}")


Claim frequency model (Poisson GLM) applied
Mean lambda: 0.1618
Min lambda: 0.0474, Max lambda: 0.3535
TPO mean lambda: 0.0869
Comp mean lambda: 0.1933
TPFT mean lambda: 0.1157


In [14]:
# Claim Severity Model: Per-Peril Gamma with Policy Caps
# Peril mix follows Malaysian retail product structure:
#   Comprehensive: own damage (AD/Windscreen/Theft/Fire) + third party (TPPD/TPBI)
#   TPO: third party only (TPPD/TPBI) - structurally cheaper claims

PERIL_DIST = {
    'Comprehensive': {
        'AD': 0.58, 'Windscreen': 0.15, 'Theft': 0.08,
        'Fire': 0.04, 'TPPD': 0.12, 'TPBI': 0.03
    },
    'TPO': {
        'TPPD': 0.78, 'TPBI': 0.22
    },
    'TPFT': {
        'TPPD': 0.444, 'TPBI': 0.111,
        'Theft': 0.296, 'Fire': 0.148
    }
}

PERIL_BASE = {
    'TPBI':       {'shape': 0.35, 'scale': 70000, 'cap': float('inf')},
    'TPPD':       {'shape': 0.55, 'scale': 9000,  'cap': 3000000},
    'Windscreen': {'shape': 2.00, 'scale': 700,   'cap': 15000}
}


def sample_claim_peril(coverage_type):
    """Sample a claim peril from the product-specific mix."""
    mix = PERIL_DIST.get(coverage_type, PERIL_DIST['Comprehensive'])
    probs = np.array(list(mix.values()))
    return np.random.choice(list(mix.keys()), p=probs / probs.sum())


def generate_single_claim(coverage_type, sum_assured):
    """Draw one claim amount (RM) with peril-specific Gamma + cap."""
    peril = sample_claim_peril(coverage_type)

    if peril == 'Theft':
        shape, scale = 1.10, max(8000, min(20000, sum_assured * 0.20))
        cap = sum_assured
    elif peril == 'Fire':
        shape, scale = 0.90, max(7000, min(18000, sum_assured * 0.15))
        cap = sum_assured
    elif peril == 'AD':
        shape, scale = 0.60, max(4500, min(12000, sum_assured * 0.10))
        cap = sum_assured
    else:
        spec = PERIL_BASE[peril]
        shape, scale, cap = spec['shape'], spec['scale'], spec['cap']

    amount = np.random.gamma(shape, scale)
    return min(amount, cap), peril


def generate_claim_total(coverage_type, sum_assured, n_claims):
    """Aggregate severity across all claims in a policy-year."""
    if n_claims <= 0:
        return 0.0, ''
    total = 0.0
    perils = []
    for _ in range(int(n_claims)):
        amt, peril = generate_single_claim(coverage_type, sum_assured)
        total += amt
        perils.append(peril)
    return round(total, 2), '/'.join(perils)


print('Per-peril severity model ready:')
print('  Comprehensive perils:', list(PERIL_DIST['Comprehensive'].keys()))
print('  TPO perils:          ', list(PERIL_DIST['TPO'].keys()))
print('  TPFT perils:         ', list(PERIL_DIST['TPFT'].keys()))


Per-peril severity model ready:
  Comprehensive perils: ['AD', 'Windscreen', 'Theft', 'Fire', 'TPPD', 'TPBI']
  TPO perils:           ['TPPD', 'TPBI']
  TPFT perils:          ['TPPD', 'TPBI', 'Theft', 'Fire']


In [15]:

# Retention Model (Binomial Logit Proxy)
# Probability of renewing policy next year.
# PREMIUM_CHANGE_PCT is fed from the annual portfolio trend (see cohort-simulation).
# Note: FINAL_PREMIUM_SST now evolves yearly (loadings + NCD), but the retention
# signal remains the portfolio-level trend to keep retention behavior stable.

def compute_retention_probability(row, premium_change_pct):
    """Compute probability of renewal.

    Key drivers (priority):
    1. Premium increase (highest sensitivity)
    2. Claim occurrence
    3. NCD level (incentive to stay)
    """
    p = 0.80  # Base renewal rate

    # Premium increase sensitivity (highest priority)
    if premium_change_pct > 0.15:
        p -= 0.15
    elif premium_change_pct > 0.05:
        p -= 0.10
    elif premium_change_pct < -0.05:
        p += 0.05  # Discounts improve retention

    # Claim occurrence effect
    p -= 0.25 if row.get('CLAIM_OCCURRED', False) else 0

    # NCD incentive to stay
    if row['NCD_YEARS'] >= 3:
        p += 0.15
    elif row['NCD_YEARS'] >= 2:
        p += 0.08

    return np.clip(p, 0.1, 0.95)


In [16]:

# Cohort Evolution Simulation (5-year forward)
# In-force policies age each year (DRIVER_AGE, CAR_AGE +1); claim frequency and
# final premium are recomputed annually with the new ages and NCD (one-year lag:
# year N is priced with the NCD earned through year N-1).

def simulate_cohort(df_initial, n_years=5, new_entrants_per_year=5000,
                    premium_trend_annual=1.06, seed=42):
    """Simulate cohort evolution.

    Args:
        df_initial: Starting cohort (Year 1)
        n_years: Number of years to simulate
        new_entrants_per_year: New policies entering each year
        premium_trend_annual: Annual premium inflation used for retention only
        seed: Random seed for reproducibility

    Returns:
        pd.DataFrame with all policy-year records
    """
    np.random.seed(seed)
    history = []
    df_active = df_initial.copy()
    # Entrants come from generate_dataset() (same assumptions as the initial cohort)

    for year_offset in range(n_years):
        year = COHORT_YEAR + year_offset
        df_active['SIM_YEAR'] = year

        # Age in-force policies (same POLID, older driver + older car).
        # Policies with COHORT_YEAR == year are brand-new entrants: keep fresh ages.
        aging_mask = df_active['COHORT_YEAR'] < year
        if aging_mask.any():
            df_active.loc[aging_mask, 'DRIVER_AGE'] += 1
            df_active.loc[aging_mask, 'CAR_AGE'] = np.minimum(
                df_active.loc[aging_mask, 'CAR_AGE'] + 1, 10
            )
            # DRIVER_AGE_CAT frozen at inception (age_to_cat removed)

        # Recompute frequency + premium with current ages and NCD
        # (NCD_LEVEL here still reflects claims through the PRIOR year -> one-year lag)
        df_active['CLAIM_LAMBDA'] = df_active.apply(compute_claim_lambda, axis=1)
        df_active['TOTAL_LOADING'] = df_active.apply(
            lambda r: total_loading(r['DRIVER_AGE_CAT'], r['CAR_AGE']), axis=1
        )
        df_active['NCD_LEVEL_PRICED'] = df_active['NCD_LEVEL']
        df_active['FINAL_PREMIUM_SST'] = df_active.apply(compute_final_premium, axis=1)

        # Simulate claims (Poisson frequency)
        df_active['CLAIM_COUNT'] = df_active['CLAIM_LAMBDA'].apply(
            lambda l: np.random.poisson(l)
        )
        df_active['CLAIM_OCCURRED'] = df_active['CLAIM_COUNT'] > 0

        # Simulate severity (per-peril Gamma, aggregated per policy-year)
        df_active['CLAIM_AMOUNT'] = 0.0
        df_active['CLAIM_PERIL'] = ''
        has_claims = df_active['CLAIM_OCCURRED']
        if has_claims.any():
            claimers = df_active.loc[has_claims]
            totals = claimers.apply(
                lambda r: generate_claim_total(
                    r['COVERAGE_TYPE'], r['SUM_ASSURED'], r['CLAIM_COUNT']
                ),
                axis=1
            )
            df_active.loc[has_claims, 'CLAIM_AMOUNT'] = [t[0] for t in totals]
            df_active.loc[has_claims, 'CLAIM_PERIL'] = [t[1] for t in totals]

        # Premium change signal (retention only - not stored premium)
        df_active['PREMIUM_CHANGE_PCT'] = premium_trend_annual ** year_offset - 1

        # Compute retention probability
        df_active['RENEWAL_PROB'] = df_active.apply(
            lambda r: compute_retention_probability(r, r['PREMIUM_CHANGE_PCT']),
            axis=1
        )

        # Simulate renewals
        df_active['RENEWED'] = (
            np.random.random(len(df_active)) < df_active['RENEWAL_PROB']
        )

        # Update NCD based on claims
        df_active.loc[~df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] += 1
        df_active.loc[df_active['CLAIM_OCCURRED'], 'NCD_YEARS'] = 0

        # Map NCD_YEARS to NCD_LEVEL
        df_active['NCD_LEVEL'] = df_active['NCD_YEARS'].apply(
            lambda yrs: NCD_TABLE.get(min(yrs, 6), 0.55)
        )

        # Record full year state (including lapsers) for retention analysis
        cols_to_keep = ['POLID', 'COVERAGE_TYPE', 'SUM_ASSURED', 'REGION',
                        'VEHICLE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
                        'CAR_AGE', 'DRIVER_GENDER', 'FLOOD_RISK', 'THEFT_RISK',
                        'BASIC_PREMIUM', 'FINAL_PREMIUM_SST', 'TOTAL_LOADING',
                        'NCD_LEVEL_PRICED', 'NCD_LEVEL',
                        'NCD_YEARS', 'CLAIM_LAMBDA',
                        'SIM_YEAR', 'CLAIM_COUNT', 'CLAIM_OCCURRED', 'CLAIM_AMOUNT',
                        'CLAIM_PERIL', 'PREMIUM_CHANGE_PCT',
                        'RENEWAL_PROB', 'RENEWED', 'COHORT_YEAR']
        history.append(df_active[cols_to_keep].copy())

        print(f"Year {year}: {len(df_active)} active policies, "
              f"claims: {df_active['CLAIM_COUNT'].sum()}, "
              f"freq: {df_active['CLAIM_OCCURRED'].mean():.1%}, "
              f"avg NCD priced: {df_active['NCD_LEVEL_PRICED'].mean():.2%}, "
              f"avg premium: RM{df_active['FINAL_PREMIUM_SST'].mean():.2f}, "
              f"retention: {df_active['RENEWED'].mean():.1%}")

        # Add new entrants for next year (same generator + assumptions as initial)
        if year_offset < n_years - 1 and new_entrants_per_year > 0:
            new_cohort = generate_dataset(
                COHORT_CONFIG, seed=seed + year_offset,
                n=new_entrants_per_year, cohort_year=year + 1,
                polid_prefix='ENT'
            )
            df_active = pd.concat(
                [df_active[df_active['RENEWED']], new_cohort],
                ignore_index=True
            )
        else:
            df_active = df_active[df_active['RENEWED']].copy()

    return pd.concat(history, ignore_index=True)


# Run simulation
cohort_results = simulate_cohort(df, n_years = 20 , new_entrants_per_year = int(0.50 * num_dataset))
print(f"\nSimulation complete. Total records: {len(cohort_results)}")
print(f"Year range: {cohort_results['SIM_YEAR'].min()} - {cohort_results['SIM_YEAR'].max()}")


Year 2026: 100000 active policies, claims: 16031, freq: 14.7%, avg NCD priced: 24.78%, avg premium: RM1552.67, retention: 82.5%


Year 2027: 132488 active policies, claims: 21078, freq: 14.6%, avg NCD priced: 30.22%, avg premium: RM1468.62, retention: 74.1%


Year 2028: 148168 active policies, claims: 23702, freq: 14.6%, avg NCD priced: 32.41%, avg premium: RM1436.26, retention: 75.0%


Year 2029: 161159 active policies, claims: 25432, freq: 14.4%, avg NCD priced: 33.90%, avg premium: RM1413.25, retention: 70.8%


Year 2030: 164021 active policies, claims: 25982, freq: 14.5%, avg NCD priced: 34.74%, avg premium: RM1399.75, retention: 70.7%


...

Year 2042: 171959 active policies, claims: 26669, freq: 14.2%, avg NCD priced: 35.69%, avg premium: RM1385.63, retention: 70.9%


Year 2043: 171956 active policies, claims: 26585, freq: 14.1%, avg NCD priced: 35.81%, avg premium: RM1381.59, retention: 70.9%


Year 2044: 171943 active policies, claims: 26470, freq: 14.1%, avg NCD priced: 35.82%, avg premium: RM1376.27, retention: 71.0%


Year 2045: 172146 active policies, claims: 26750, freq: 14.2%, avg NCD priced: 35.85%, avg premium: RM1374.84, retention: 70.9%



Simulation complete. Total records: 3263823
Year range: 2026 - 2045


In [17]:
# Cohort Analysis Summary

def analyze_cohort(df_cohort):
    """Generate summary statistics from cohort simulation."""
    summary = []
    
    for year in sorted(df_cohort['SIM_YEAR'].unique()):
        year_df = df_cohort[df_cohort['SIM_YEAR'] == year]
        summary.append({
            'Year': year,
            'Active_Policies': len(year_df),
            'Total_Claims': year_df['CLAIM_COUNT'].sum(),
            'Avg_Claims_Per_Policy': year_df['CLAIM_COUNT'].mean(),
            'Total_Claim_Amount': year_df['CLAIM_AMOUNT'].sum(),
            'Avg_Claim_Amount': year_df.loc[
                year_df['CLAIM_COUNT'] > 0, 'CLAIM_AMOUNT'
            ].mean() if year_df['CLAIM_COUNT'].sum() > 0 else 0,
            'Avg_NCD_Level': year_df['NCD_LEVEL'].mean(),
            'Retention_Rate': year_df['RENEWED'].mean()
            if 'RENEWED' in year_df.columns else float('nan'),
            'Avg_Final_Premium': year_df['FINAL_PREMIUM_SST'].mean()
        })
    
    return pd.DataFrame(summary)

# Generate and display cohort summary
cohort_summary = analyze_cohort(cohort_results)
print('=== 5-Year Cohort Evolution Summary ===\n')
print(cohort_summary.to_string(index=False))
print(f"\n=== Key Findings ===")
print(f"Total claims over 5 years: {cohort_results['CLAIM_COUNT'].sum():.0f}")
print(f"Total claim cost: RM{cohort_results['CLAIM_AMOUNT'].sum():,.2f}")
print(f"Final avg NCD level: {cohort_results[cohort_results['SIM_YEAR']==2028]['NCD_LEVEL'].mean():.2%}")
print(f"Loss ratio: {cohort_results['CLAIM_AMOUNT'].sum() / cohort_results['FINAL_PREMIUM_SST'].sum():.2%}")

=== 5-Year Cohort Evolution Summary ===

 Year  Active_Policies  Total_Claims  Avg_Claims_Per_Policy  Total_Claim_Amount  Avg_Claim_Amount  Avg_NCD_Level  Retention_Rate  Avg_Final_Premium
 2026           100000         16031               0.160310        100888600.16       6861.769718       0.313006        0.824880        1552.673702
 2027           132488         21078               0.159094        125548120.31       6505.758126       0.338415        0.740958        1468.616740
 2028           148168         23702               0.159967        148492572.08       6857.512334       0.353778        0.750223        1436.255819
 2029           161159         25432               0.157807        157390960.62       6761.360968       0.363601        0.707506        1413.252572
 2030           164021         25982               0.158407        159884786.97       6718.977432       0.368224        0.707403        1399.753847
 2031           166029         26055               0.156930        1645

In [18]:
# Test: Sample Claim Generation
# Demonstrate claim modeling on sample policies

def generate_sample_claims(df, n=5):
    """Generate sample claims for testing the model."""
    samples = df.sample(n=min(n, len(df)), random_state=42)

    print('=== Sample Claim Generation ===\n')

    for idx, row in samples.iterrows():
        lamb = row['CLAIM_LAMBDA']
        n_claims = np.random.poisson(lamb)

        print(f"Policy: {row['POLID'][:12]}...")
        print(f"  Coverage: {row['COVERAGE_TYPE']}, Vehicle: {row['VEHICLE_TYPE']}")
        print(f"  Sum Assured: RM{row['SUM_ASSURED']:,.0f}")
        print(f"  Flood Risk: {row['FLOOD_RISK']}, Theft Risk: {row['THEFT_RISK']}")
        print(f"  NCD Years: {row['NCD_YEARS']} ({row['NCD_LEVEL']:.1%})")
        print(f"  Claim Intensity (lambda): {lamb:.3f}")

        if n_claims > 0:
            total_claim, perils = generate_claim_total(
                row['COVERAGE_TYPE'], row['SUM_ASSURED'], n_claims
            )
            print(f"  Perils: {perils}")
            print(f"  TOTAL CLAIM: RM{total_claim:,.2f}")
        else:
            print('  No claims this year')

        print(f"  Basic Premium: RM{row['BASIC_PREMIUM']:,.2f}")
        print(f"  Final Premium (w/ SST): RM{row['FINAL_PREMIUM_SST']:,.2f}")
        print()


# Run sample generation
generate_sample_claims(df, n=2)


=== Sample Claim Generation ===

Policy: INIT2026-075...
  Coverage: Comprehensive, Vehicle: EV
  Sum Assured: RM72,000
  Flood Risk: False, Theft Risk: False
  NCD Years: 1 (25.0%)
  Claim Intensity (lambda): 0.239
  No claims this year
  Basic Premium: RM1,731.70
  Final Premium (w/ SST): RM1,885.20

Policy: INIT2026-080...
  Coverage: TPO, Vehicle: ICE
  Sum Assured: RM43,000
  Flood Risk: False, Theft Risk: True
  NCD Years: 1 (25.0%)
  Claim Intensity (lambda): 0.098
  No claims this year
  Basic Premium: RM67.50
  Final Premium (w/ SST): RM74.34



In [19]:

# ============================================================
# Statistical Validation (Part A: Core Tests)
# ============================================================

results = cohort_results.copy()
passed = []
failed = []

def check(name, cond, detail=''):
    if cond:
        passed.append(name)
        print(f"[PASS] {name}")
    else:
        failed.append(name)
        print(f"[FAIL] {name} {detail}")

# Test 1: Overall claim frequency in plausible band (10%-20%)
overall_freq = results['CLAIM_OCCURRED'].mean()
check('1. Overall claim frequency within 10%-20%',
      0.10 <= overall_freq <= 0.20,
      f"(actual {overall_freq:.1%})")

# Test 2: TPO expected claim cost per policy-year < Comprehensive
# (frequency x mean severity - TPO has no own-damage exposure)
comp_freq_t = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq_t = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
comp_sev = results.loc[(results['COVERAGE_TYPE']=='Comprehensive') &
                       (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
tpo_sev = results.loc[(results['COVERAGE_TYPE']=='TPO') &
                      (results['CLAIM_AMOUNT']>0), 'CLAIM_AMOUNT']
comp_mean = comp_sev.mean() if len(comp_sev) else 0
tpo_mean = tpo_sev.mean() if len(tpo_sev) else 0
comp_cost = comp_freq_t * comp_mean
tpo_cost = tpo_freq_t * tpo_mean
check('2. TPO expected claim cost/policy-year < Comprehensive',
      tpo_cost < comp_cost,
      f"(Comp RM{comp_cost:,.0f} vs TPO RM{tpo_cost:,.0f})")

# Test 3: Per-peril means (report; caps enforced at draw time)
peril_means = (results[results['CLAIM_AMOUNT']>0]
               .assign(peril_first=lambda d: d['CLAIM_PERIL'].str.split('/').str[0])
               .groupby('peril_first')['CLAIM_AMOUNT'].mean())
print('\n  Per-peril mean severity:')
for peril, m in peril_means.sort_values(ascending=False).items():
    print(f"    {peril:10s} RM{m:,.0f}  (n={len(results[results['CLAIM_PERIL'].str.contains(peril)])})")
print('  (caps enforced at draw time by construction)')

# Test 4: Loss ratio (incurred / earned premium) - report only
earned = results['FINAL_PREMIUM_SST'].sum()
incurred = results['CLAIM_AMOUNT'].sum()
loss_ratio = incurred / earned if earned > 0 else float('nan')
print(f"\n  Loss ratio: {loss_ratio:.1%} (earned RM{earned:,.0f}, incurred RM{incurred:,.0f})")
if not (0.40 <= loss_ratio <= 0.80):
    print(f"  [WARN] Loss ratio outside 40%-80% band - inspect premium adequacy")
else:
    print(f"  [OK]   Loss ratio within 40%-80% band")

# Test 5: NCD mechanics - claim resets to 0, claim-free increments
reset_ok = (results.loc[results['CLAIM_OCCURRED'], 'NCD_YEARS'] == 0).mean()
inc_ok = (results.loc[~results['CLAIM_OCCURRED'], 'NCD_YEARS'] >= 1).mean()
check('5a. Claims reset NCD_YEARS to 0', reset_ok >= 0.99,
      f"(reset rate {reset_ok:.1%})")
check('5b. Claim-free years increment NCD', inc_ok >= 0.99,
      f"(increment rate {inc_ok:.1%})")

# Test 6: TPO frequency < Comprehensive frequency
comp_freq = results.loc[results['COVERAGE_TYPE']=='Comprehensive', 'CLAIM_OCCURRED'].mean()
tpo_freq = results.loc[results['COVERAGE_TYPE']=='TPO', 'CLAIM_OCCURRED'].mean()
check('6. TPO claim frequency < Comprehensive',
      tpo_freq < comp_freq,
      f"(Comp {comp_freq:.1%} vs TPO {tpo_freq:.1%})")

# Test 7: Data integrity - no nulls/negatives in key columns
key_cols = ['CLAIM_COUNT', 'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'FINAL_PREMIUM_SST',
            'NCD_LEVEL', 'NCD_YEARS', 'RENEWED', 'CAR_AGE', 'TOTAL_LOADING',
            'NCD_LEVEL_PRICED']
null_bad = results[key_cols].isnull().sum().sum()
neg_bad = (results['CLAIM_AMOUNT'] < 0).sum() + (results['FINAL_PREMIUM_SST'] <= 0).sum()
check('7. No nulls in key columns', null_bad == 0, f"(nulls: {null_bad})")
check('7b. No negative/zero premium or negative claims', neg_bad == 0,
      f"(bad: {neg_bad})")

# Test 8b: Ageing works - in-force DRIVER_AGE/CAR_AGE increase across years
age_by_year = results.groupby('SIM_YEAR')[['DRIVER_AGE', 'CAR_AGE']].mean()
age_growth = age_by_year['DRIVER_AGE'].iloc[-1] > age_by_year['DRIVER_AGE'].iloc[0]
car_growth = age_by_year['CAR_AGE'].iloc[-1] > age_by_year['CAR_AGE'].iloc[0]
check('8b. Mean DRIVER_AGE rises across years', age_growth,
      f"({age_by_year['DRIVER_AGE'].iloc[0]:.1f} -> {age_by_year['DRIVER_AGE'].iloc[-1]:.1f})")
check('8c. Mean CAR_AGE rises across years', car_growth,
      f"({age_by_year['CAR_AGE'].iloc[0]:.1f} -> {age_by_year['CAR_AGE'].iloc[-1]:.1f})")

# Test 9: CAR_AGE plausibility
# Fleet age at inception (year-1 policies), not the aged 20-yr book
car_age_med = results.loc[results['SIM_YEAR'] == COHORT_YEAR, 'CAR_AGE'].median()
check('9. CAR_AGE median within 3-4 years', 3.0 <= car_age_med <= 4.0,
      f"(median {car_age_med:.1f})")
check('9b. CAR_AGE within [0,10]', results['CAR_AGE'].between(0, 10).all())
genz_car = results.loc[results['DRIVER_AGE_CAT'] == 'Gen-Z', 'CAR_AGE'].mean()
other_car = results.loc[results['DRIVER_AGE_CAT'] != 'Gen-Z', 'CAR_AGE'].mean()
check('9c. Gen-Z drive newer cars than other cohorts', genz_car < other_car,
      f"(Gen-Z {genz_car:.1f} vs others {other_car:.1f})")

# Test 10: Premium evolves across years for in-force policies
years_per_polid = results.groupby('POLID')['SIM_YEAR'].nunique()
multi_year = results[results['POLID'].isin(
    years_per_polid[years_per_polid >= 3].index
)]
prem_levels = multi_year.groupby('POLID')['FINAL_PREMIUM_SST'].nunique()
prem_varies = (prem_levels > 1).mean()
check('10. Premium varies across years for multi-year policies',
      prem_varies >= 0.9, f"({prem_varies:.1%} of policies vary)")

print(f"\n===== VALIDATION RESULT: {len(passed)} passed, {len(failed)} failed =====")


[PASS] 1. Overall claim frequency within 10%-20%
[PASS] 2. TPO expected claim cost/policy-year < Comprehensive



  Per-peril mean severity:


    TPBI       RM25,389  (n=29493)
    Theft      RM13,400  (n=53417)


    Fire       RM9,101  (n=26810)


    TPPD       RM5,443  (n=112389)


    AD         RM4,298  (n=214467)


    Windscreen RM1,936  (n=57664)
  (caps enforced at draw time by construction)

  Loss ratio: 69.4% (earned RM4,562,529,414, incurred RM3,164,378,774)
  [OK]   Loss ratio within 40%-80% band
[PASS] 5a. Claims reset NCD_YEARS to 0
[PASS] 5b. Claim-free years increment NCD
[PASS] 6. TPO claim frequency < Comprehensive


[PASS] 7. No nulls in key columns
[PASS] 7b. No negative/zero premium or negative claims
[PASS] 8b. Mean DRIVER_AGE rises across years
[PASS] 8c. Mean CAR_AGE rises across years
[PASS] 9. CAR_AGE median within 3-4 years
[PASS] 9b. CAR_AGE within [0,10]
[PASS] 9c. Gen-Z drive newer cars than other cohorts


[PASS] 10. Premium varies across years for multi-year policies

===== VALIDATION RESULT: 13 passed, 0 failed =====


In [20]:

# ============================================================
# FINDING 1: TPO Underpricing (documented, deliberately NOT modelled away)
# ============================================================
tpo = results.loc[results['COVERAGE_TYPE'] == 'TPO']
comp = results.loc[results['COVERAGE_TYPE'] == 'Comprehensive']
tpo_earned = tpo['FINAL_PREMIUM_SST'].sum()
tpo_incurred = tpo['CLAIM_AMOUNT'].sum()
tpo_lr = tpo_incurred / tpo_earned if tpo_earned > 0 else float('nan')
comp_earned = comp['FINAL_PREMIUM_SST'].sum()
comp_lr = comp['CLAIM_AMOUNT'].sum() / comp_earned if comp_earned > 0 else float('nan')

print('=== FINDING 1: TPO gross loss ratio ===')
print(f"TPO gross LR : {tpo_lr:.1%}  (earned RM{tpo_earned:,.0f}, incurred RM{tpo_incurred:,.0f})")
print(f"Comp gross LR: {comp_lr:.1%}")
print()
print('Root cause: TPO premium is a fixed flat amount from sources/rates.csv')
print('  (RM74-227, independent of SUM_ASSURED), while TPO claims come from the')
print('  SA-free severity model with a heavy TPBI tail (unlimited bodily injury).')
print('  => pricing inadequacy in the RATE FILE, not a claims-model defect.')
print()
print('Resolution: OUT OF SCOPE - reprice TPO Fixed Amounts in rates.csv.')
print('Deliberate choice: no artificial decline threshold, no model-side reprice,')
print('  and no tampering with the rate file.')


=== FINDING 1: TPO gross loss ratio ===
TPO gross LR : 591.6%  (earned RM68,168,297, incurred RM403,291,105)
Comp gross LR: 55.1%

Root cause: TPO premium is a fixed flat amount from sources/rates.csv
  (RM74-227, independent of SUM_ASSURED), while TPO claims come from the
  SA-free severity model with a heavy TPBI tail (unlimited bodily injury).
  => pricing inadequacy in the RATE FILE, not a claims-model defect.

Resolution: OUT OF SCOPE - reprice TPO Fixed Amounts in rates.csv.
Deliberate choice: no artificial decline threshold, no model-side reprice,
  and no tampering with the rate file.


In [21]:

# ============================================================
# Statistical Validation (Part B) + EDA Enrichment
# ============================================================

# Test 8: Reproducibility - re-running with same seed gives identical claims
np.random.seed(999)
run_a = simulate_cohort(df, n_years=3, seed=7)
np.random.seed(999)
run_b = simulate_cohort(df, n_years=3, seed=7)
same_claims = (run_a['CLAIM_COUNT'].values == run_b['CLAIM_COUNT'].values).all()
same_amounts = np.allclose(run_a['CLAIM_AMOUNT'].values, run_b['CLAIM_AMOUNT'].values)
if same_claims and same_amounts:
    print("[PASS] 8. Reproducibility: same seed -> identical claims")
else:
    print("[FAIL] 8. Reproducibility: seeded runs differ")

# ---- EDA enrichment ----
res = cohort_results.copy()

# E1: Loss ratio by coverage type (n/a if no earned premium)
def lr_ratio(d):
    earned = d['FINAL_PREMIUM_SST'].sum()
    if earned <= 0:
        return 'n/a (no earned premium)'
    return f"{d['CLAIM_AMOUNT'].sum() / earned:.1%}"
lr_by_cov = res.groupby('COVERAGE_TYPE').apply(lr_ratio)
print('\nE1. Loss ratio by coverage type (TPO underpriced - see FINDING 1):')
print(lr_by_cov.to_string())

# E2: Peril mix across all claims
peril_counts = res[res['CLAIM_AMOUNT'] > 0]['CLAIM_PERIL'].str.split('/').explode().value_counts()
print('\nE2. Peril mix:')
print(peril_counts.to_string())

# E3: Claim frequency by age category
freq_by_age = res.groupby('DRIVER_AGE_CAT')['CLAIM_OCCURRED'].mean().sort_values(ascending=False)
print('\nE3. Claim frequency by age category:')
print(freq_by_age.map(lambda x: f"{x:.1%}").to_string())

# E4: Retention by claim status
ret_by_claim = res.groupby('CLAIM_OCCURRED')['RENEWED'].mean()
print('\nE4. Retention by claim status:')
print(ret_by_claim.map(lambda x: f"{x:.1%}").to_string())


Year 2026: 100000 active policies, claims: 16247, freq: 14.8%, avg NCD priced: 24.78%, avg premium: RM1552.67, retention: 82.4%


Year 2027: 87381 active policies, claims: 14119, freq: 14.8%, avg NCD priced: 32.84%, avg premium: RM1423.40, retention: 74.4%


Year 2028: 70023 active policies, claims: 10975, freq: 14.4%, avg NCD priced: 36.64%, avg premium: RM1371.31, retention: 76.8%


Year 2026: 100000 active policies, claims: 16247, freq: 14.8%, avg NCD priced: 24.78%, avg premium: RM1552.67, retention: 82.4%


Year 2027: 87381 active policies, claims: 14119, freq: 14.8%, avg NCD priced: 32.84%, avg premium: RM1423.40, retention: 74.4%


Year 2028: 70023 active policies, claims: 10975, freq: 14.4%, avg NCD priced: 36.64%, avg premium: RM1371.31, retention: 76.8%
[PASS] 8. Reproducibility: same seed -> identical claims



E1. Loss ratio by coverage type (TPO underpriced - see FINDING 1):
COVERAGE_TYPE
Comprehensive     55.1%
TPFT              88.1%
TPO              591.6%



E2. Peril mix:
CLAIM_PERIL
AD            227300
TPPD          114973
Windscreen     58614
Theft          54070
TPBI           29599
Fire           27000

E3. Claim frequency by age category:
DRIVER_AGE_CAT
Gen-Z         17.3%
Senior        16.2%
Boomers       12.5%
Millennial    12.0%

E4. Retention by claim status:
CLAIM_OCCURRED
False    75.3%
True     49.4%


In [22]:

# ============================================================
# Individual Policy Trajectories (premium evolution over years)
# ============================================================

def plot_policy_trajectories(df_cohort, n_policies=5, min_years=3,
                             out='images/policy-trajectories.png', seed=42):
    """Plot premium + priced-NCD trajectories for random multi-year policies."""
    import os
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    years_per_polid = df_cohort.groupby('POLID')['SIM_YEAR'].nunique()
    eligible = years_per_polid[years_per_polid >= min_years].index.tolist()
    rng = np.random.RandomState(seed)
    picks = [str(p) for p in rng.choice(
        eligible, size=min(n_policies, len(eligible)), replace=False
    )]

    sel = df_cohort[df_cohort['POLID'].isin(picks)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for pid in picks:
        p = sel[sel['POLID'] == pid].sort_values('SIM_YEAR')
        axes[0].plot(p['SIM_YEAR'], p['FINAL_PREMIUM_SST'], marker='o',
                     label=f"{pid[:12]}...")
        axes[1].plot(p['SIM_YEAR'], p['NCD_LEVEL_PRICED'], marker='s')
    axes[0].set_title('Final premium by year')
    axes[0].set_xlabel('SIM_YEAR')
    axes[0].set_ylabel('FINAL_PREMIUM_SST (RM)')
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[1].set_title('NCD used for pricing by year')
    axes[1].set_xlabel('SIM_YEAR')
    axes[1].set_ylabel('NCD_LEVEL_PRICED')
    axes[1].grid(alpha=0.3)
    fig.tight_layout()

    os.makedirs(os.path.dirname(out), exist_ok=True)
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Trajectory plot saved: {out}")

    print('\n=== Selected policy trajectories (year-by-year) ===')
    cols = ['POLID', 'SIM_YEAR', 'COVERAGE_TYPE', 'DRIVER_AGE_CAT', 'DRIVER_AGE',
            'CAR_AGE', 'NCD_LEVEL_PRICED', 'FINAL_PREMIUM_SST', 'CLAIM_OCCURRED']
    print(sel.sort_values(['POLID', 'SIM_YEAR'])[cols].to_string(index=False))


plot_policy_trajectories(cohort_results)


Trajectory plot saved: images/policy-trajectories.png

=== Selected policy trajectories (year-by-year) ===
         POLID  SIM_YEAR COVERAGE_TYPE DRIVER_AGE_CAT  DRIVER_AGE  CAR_AGE  NCD_LEVEL_PRICED  FINAL_PREMIUM_SST  CLAIM_OCCURRED
ENT2027-010141      2027 Comprehensive     Millennial          40        4            0.0000            2544.24           False
ENT2027-010141      2028 Comprehensive     Millennial          41        5            0.2500            1959.29           False
ENT2027-010141      2029 Comprehensive     Millennial          42        6            0.3000            1876.37           False
ENT2027-010141      2030 Comprehensive     Millennial          43        7            0.3833            1695.11           False
ENT2027-010141      2031 Comprehensive     Millennial          44        8            0.4500            1549.26           False
ENT2027-010141      2032 Comprehensive     Millennial          45        9            0.5500            1298.24            Tr

In [23]:
# ============================================================================
# OVERTHINKER-STYLE VALIDATION (Part A + Enhanced Tests + Correlation)
# Adapted from reference overthinker_gen_data.ipynb Section 6
# ============================================================================
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import kstest, gamma as gamma_dist, normaltest, norm

final_dataset = cohort_results.copy()

print("="*70)
print("DATASET VALIDATION")
print("="*70)

validation_results = []

# 1. Premium vs Sum Insured Correlation (Comprehensive; TPO premium is SA-free by design)
comp_sub = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']
corr_comp = comp_sub['FINAL_PREMIUM_SST'].corr(comp_sub['SUM_ASSURED'])
validation_results.append({
    'Test': 'Premium vs Sum Insured Correlation (Comp)',
    'Value': f"{corr_comp:.3f}",
    'Expected': '> 0.5',
    'Pass': corr_comp > 0.5
})

# 2. Young Driver Premium Loading
young_avg = final_dataset[final_dataset['DRIVER_AGE'] < 25]['FINAL_PREMIUM_SST'].mean()
mature_avg = final_dataset[final_dataset['DRIVER_AGE'].between(30, 50)]['FINAL_PREMIUM_SST'].mean()
loading = young_avg / mature_avg
validation_results.append({
    'Test': 'Young Driver Premium Loading',
    'Value': f"{loading:.2f}x",
    'Expected': '> 1.05x (driver loading partly offset by newer cars)',
    'Pass': loading > 1.05
})

# 3. Claim Frequency Range
claim_freq = (final_dataset['CLAIM_COUNT'] > 0).mean()
validation_results.append({
    'Test': 'Overall Claim Frequency',
    'Value': f"{claim_freq*100:.2f}%",
    'Expected': '10-20%',
    'Pass': 0.10 <= claim_freq <= 0.20
})

# 4. NCD Progression
ncd_2026 = final_dataset[final_dataset['SIM_YEAR'] == 2026]['NCD_LEVEL_PRICED'].mean()
ncd_2030 = final_dataset[final_dataset['SIM_YEAR'] == 2030]['NCD_LEVEL_PRICED'].mean()
validation_results.append({
    'Test': 'NCD Progression (2026 < 2030)',
    'Value': f"{ncd_2026*100:.1f}% to {ncd_2030*100:.1f}%",
    'Expected': 'Increasing',
    'Pass': ncd_2030 > ncd_2026
})

# 5. Loss Ratio Range
loss_ratio = final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()
validation_results.append({
    'Test': 'Overall Loss Ratio',
    'Value': f"{loss_ratio:.2%}",
    'Expected': '50-80%',
    'Pass': 0.50 <= loss_ratio <= 0.80
})

# 6. Region premium difference (report only - rate file dependent)
east_avg = final_dataset[final_dataset['REGION'].str.contains('East')]['FINAL_PREMIUM_SST'].mean()
pen_avg = final_dataset[final_dataset['REGION'].str.contains('Peninsular')]['FINAL_PREMIUM_SST'].mean()
region_ratio = east_avg / pen_avg if pen_avg > 0 else float('nan')
print(f"  Region premium ratio (East/Peninsular): {region_ratio:.2f}x (report only)")

# 7. Comprehensive vs TPO Premium
comp_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['FINAL_PREMIUM_SST'].mean()
tpo_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['FINAL_PREMIUM_SST'].mean()
comp_ratio = comp_avg / tpo_avg if tpo_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comprehensive Premium > TPO',
    'Value': f"{comp_ratio:.2f}x",
    'Expected': '> 1.8x',
    'Pass': comp_ratio > 1.8
})

# 7b. TPFT premium between TPO and Comprehensive
tpft_avg = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['FINAL_PREMIUM_SST'].mean()
validation_results.append({
    'Test': 'TPFT Premium between TPO & Comp',
    'Value': f"RM{tpo_avg:,.0f} < RM{tpft_avg:,.0f} < RM{comp_avg:,.0f}",
    'Expected': 'TPO < TPFT < Comprehensive',
    'Pass': tpo_avg < tpft_avg < comp_avg
})

# 8. NCD Discount (controlled: same policy priced at NCD=0 vs actual)
def premium_at_zero_ncd(row):
    r = row.copy()
    r['NCD_LEVEL'] = 0.0
    return compute_final_premium(r)

sample = final_dataset.sample(min(30000, len(final_dataset)), random_state=42).copy()
sample['PREMIUM_NCD0'] = sample.apply(premium_at_zero_ncd, axis=1)
sample['NCD_DISCOUNT'] = 1 - sample['FINAL_PREMIUM_SST'] / sample['PREMIUM_NCD0']
max_ncd_disc = sample.loc[sample['NCD_LEVEL_PRICED'] == 0.55, 'NCD_DISCOUNT'].median()
validation_results.append({
    'Test': 'NCD Discount at max tier (controlled)',
    'Value': f"{max_ncd_disc*100:.1f}%",
    'Expected': '45-60%',
    'Pass': 0.45 <= max_ncd_disc <= 0.60
})

# 9. Comprehensive Premium Trend (2026 -> 2030)
comp_2026_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2026) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_2030_avg = final_dataset[(final_dataset['SIM_YEAR'] == 2030) & (final_dataset['COVERAGE_TYPE'] == 'Comprehensive')]['FINAL_PREMIUM_SST'].mean()
comp_trend = comp_2030_avg / comp_2026_avg if comp_2026_avg > 0 else float('nan')
validation_results.append({
    'Test': 'Comp Premium Trend (2026 to 2030)',
    'Value': f"RM{comp_2026_avg:,.0f} to RM{comp_2030_avg:,.0f} ({comp_trend:.2f}x)",
    'Expected': 'Mild softening (0.80x-0.99x)',
    'Pass': 0.80 <= comp_trend < 1.00
})

validation_df = pd.DataFrame(validation_results)
print("\n" + validation_df.to_string(index=False))
all_pass = validation_df['Pass'].all()
print("\n" + "="*70)
if all_pass:
    print("ALL VALIDATION TESTS PASSED")
else:
    print("SOME VALIDATION TESTS FAILED - REVIEW ABOVE")
print("="*70)

# ---- Correlation Heatmap ----
numeric_cols = ['DRIVER_AGE', 'CAR_AGE', 'SUM_ASSURED', 'FINAL_PREMIUM_SST',
                'TOTAL_LOADING', 'NCD_LEVEL_PRICED', 'CLAIM_COUNT',
                'CLAIM_AMOUNT', 'CLAIM_LAMBDA', 'NCD_YEARS']
correlation_matrix = final_dataset[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Correlation Matrix - Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nCorrelation heatmap saved: images/correlation_heatmap.png")

# ============================================================================
# PART B: ENHANCED STATISTICAL TESTS
# ============================================================================
print("\n" + "="*70)
print("ENHANCED STATISTICAL VALIDATION")
print("="*70)

enhanced_results = []

# 1. Premium Log-Normal Fit (KS)
log_premiums = np.log(final_dataset['FINAL_PREMIUM_SST'])
ks_stat, ks_pval = kstest(log_premiums, norm(loc=log_premiums.mean(), scale=log_premiums.std()).cdf)
enhanced_results.append({
    'Test': 'Premium Log-Normal Fit (KS test)',
    'Statistic': f"KS={ks_stat:.4f}",
    'P-value': f"{ks_pval:.4e}",
    'Interpretation': 'Approx log-normal' if ks_stat < 0.1 else 'Mixture (TPO+Comp+NCD)',
    'Pass': ks_stat < 0.20
})
print("\n1. PREMIUM DISTRIBUTION FIT TEST (Log-Normal)")
print(f"   KS Statistic: {ks_stat:.4f}, p-value: {ks_pval:.4e}")
print(f"   (N={len(final_dataset):,}; threshold 0.15 given coverage/NCD mixture)")

# 2. Severity Gamma Fit by coverage
print("\n2. CLAIM SEVERITY DISTRIBUTION FIT (Gamma)")
print("-"*60)
for cov_type in ['Comprehensive', 'TPFT', 'TPO']:
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 30:
        shape_fit, loc_fit, scale_fit = gamma_dist.fit(claims_subset, floc=0)
        ks_s, ks_p = kstest(claims_subset, gamma_dist(a=shape_fit, loc=loc_fit, scale=scale_fit).cdf)
        enhanced_results.append({
            'Test': f'Severity Gamma Fit ({cov_type})',
            'Statistic': f"shape={shape_fit:.2f}, KS={ks_s:.4f}",
            'P-value': f"{ks_p:.4e}",
            'Interpretation': f'Per-peril mixture, shape={shape_fit:.2f}',
            'Pass': ks_s < 0.20
        })
        print(f"   {cov_type}: shape={shape_fit:.2f}, scale={scale_fit:.0f}, KS={ks_s:.4f}")
    else:
        print(f"   {cov_type}: Insufficient claims (n={len(claims_subset)})")

# 3. NCD Distribution by Year
print("\n3. NCD DISTRIBUTION BY YEAR")
print("-"*60)
ncd_by_year = final_dataset.groupby('SIM_YEAR')['NCD_LEVEL_PRICED'].value_counts(normalize=True).unstack(fill_value=0)
ncd_by_year = ncd_by_year.reindex(columns=sorted(ncd_by_year.columns))
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    yd = final_dataset[final_dataset['SIM_YEAR'] == year]
    print(f"   {year}: Avg NCD={yd['NCD_LEVEL_PRICED'].mean()*100:.1f}% | NCD=0%: {(yd['NCD_LEVEL_PRICED']==0.0).mean()*100:.1f}% | NCD=55%: {(yd['NCD_LEVEL_PRICED']==0.55).mean()*100:.1f}%")

ncd_2026_max = (final_dataset[final_dataset['SIM_YEAR']==2026]['NCD_LEVEL_PRICED']==0.55).mean()
ncd_2030_max = (final_dataset[final_dataset['SIM_YEAR']==2030]['NCD_LEVEL_PRICED']==0.55).mean()
enhanced_results.append({
    'Test': 'NCD: 55% tier grows over years',
    'Statistic': f"{ncd_2026_max*100:.1f}% to {ncd_2030_max*100:.1f}%",
    'P-value': '-',
    'Interpretation': 'Loyal claim-free policies accumulate',
    'Pass': ncd_2030_max > ncd_2026_max
})

# NCD distribution plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ncd_labels = sorted(final_dataset['NCD_LEVEL_PRICED'].unique())
ncd_year_data = []
for year in sorted(final_dataset['SIM_YEAR'].unique()):
    year_ncd = final_dataset[final_dataset['SIM_YEAR'] == year]['NCD_LEVEL_PRICED']
    row = {f'{n*100:.0f}%': (year_ncd == n).mean()*100 for n in ncd_labels}
    row['Year'] = year
    ncd_year_data.append(row)
ncd_plot_df = pd.DataFrame(ncd_year_data).set_index('Year')
ncd_plot_df.plot(kind='bar', stacked=True, ax=axes[0], colormap='YlOrRd_r')
axes[0].set_title('NCD Distribution by Year', fontsize=14, fontweight='bold')
axes[0].set_ylabel('% of Policies')
axes[0].legend(title='NCD %', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[0].set_xlabel('SIM_YEAR')

tier_labels = {0.0: '0%', 0.25: '25%', 0.30: '30%', 0.3833: '38%', 0.45: '45%', 0.55: '55%'}
final_dataset['NCD_TIER'] = final_dataset['NCD_LEVEL_PRICED'].map(tier_labels)
final_dataset.boxplot(column='FINAL_PREMIUM_SST', by='NCD_TIER', ax=axes[1])
axes[1].set_title('Premium Distribution by NCD Tier', fontsize=14, fontweight='bold')
axes[1].set_xlabel('NCD Tier')
axes[1].set_ylabel('Premium (RM)')
axes[1].get_figure().suptitle('')
plt.tight_layout()
plt.savefig('images/ncd_validation.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Loss Ratio by Segment
print("\n4. LOSS RATIO BY SEGMENT")
print("-"*60)
total_premium = final_dataset['FINAL_PREMIUM_SST'].sum()
total_incurred = final_dataset['CLAIM_AMOUNT'].sum()
overall_lr = total_incurred / total_premium
print("   By Coverage Type:")
for cov in ['Comprehensive', 'TPFT', 'TPO']:
    subset = final_dataset[final_dataset['COVERAGE_TYPE'] == cov]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {cov:14s}: LR={lr:.2%} (n={len(subset):,})")
final_dataset['age_band'] = pd.cut(final_dataset['DRIVER_AGE'],
                                   bins=[17, 25, 35, 50, 65, 100],
                                   labels=['18-25', '26-35', '36-50', '51-65', '66+'])
print("   By Age Band:")
for band in ['18-25', '26-35', '36-50', '51-65', '66+']:
    subset = final_dataset[final_dataset['age_band'] == band]
    if len(subset) > 0 and subset['FINAL_PREMIUM_SST'].sum() > 0:
        lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
        print(f"     {band:6s}: LR={lr:.2%} (n={len(subset):,})")
print("   By Region:")
for loc in ['Peninsular Malaysia', 'East Malaysia (Sabah, Sawarak & Labuan)']:
    subset = final_dataset[final_dataset['REGION'] == loc]
    lr = subset['CLAIM_AMOUNT'].sum() / subset['FINAL_PREMIUM_SST'].sum()
    print(f"     {loc[:12]:12s}: LR={lr:.2%}")
enhanced_results.append({
    'Test': 'Loss Ratio in Actuarial Range',
    'Statistic': f"LR={overall_lr:.2%}",
    'P-value': '-',
    'Interpretation': 'Typically 50-80% for motor',
    'Pass': 0.40 <= overall_lr <= 0.90
})

# 5. Severity Percentile Validation
print("\n5. SEVERITY PERCENTILE VALIDATION")
print("-"*60)
severity_targets = {
    'Comprehensive': {'mean': (4000, 10000), 'p95': (15000, 60000)},
    'TPO': {'mean': (8000, 30000), 'p95': (25000, 600000)},
    'TPFT': {'mean': (5000, 15000), 'p95': (20000, 120000)}
}
for cov_type, targets in severity_targets.items():
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov_type) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        mean_sev = claims_subset.mean()
        p95_sev = claims_subset.quantile(0.95)
        mean_pass = targets['mean'][0] <= mean_sev <= targets['mean'][1]
        p95_pass = targets['p95'][0] <= p95_sev <= targets['p95'][1]
        print(f"   {cov_type}: Mean=RM{mean_sev:,.0f} (target RM{targets['mean'][0]:,}-{targets['mean'][1]:,}) {'OK' if mean_pass else 'CHECK'}")
        print(f"     P95=RM{p95_sev:,.0f} (target RM{targets['p95'][0]:,}-{targets['p95'][1]:,}) {'OK' if p95_pass else 'CHECK'}")
        enhanced_results.append({'Test': f'Severity Mean ({cov_type})', 'Statistic': f"RM{mean_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["mean"][0]:,}-{targets["mean"][1]:,}', 'Pass': mean_pass})
        enhanced_results.append({'Test': f'Severity P95 ({cov_type})', 'Statistic': f"RM{p95_sev:,.0f}", 'P-value': '-', 'Interpretation': f'Target RM{targets["p95"][0]:,}-{targets["p95"][1]:,}', 'Pass': p95_pass})

# 6. Longitudinal Consistency (POLID tracking)
print("\n6. LONGITUDINAL CONSISTENCY")
print("-"*60)
ph_counts = final_dataset.groupby('POLID')['SIM_YEAR'].nunique()
print(f"   Unique policies: {len(ph_counts):,}")
print(f"   Policies with 1 year: {(ph_counts == 1).sum():,}")
print(f"   Policies with 2+ years: {(ph_counts >= 2).sum():,}")
print(f"   Policies with all 5 years: {(ph_counts == 5).sum():,}")
multi_year_phs = ph_counts[ph_counts >= 2].index[:1000]
age_errors = 0
car_errors = 0
ncd_reset_failures = 0
total_claims_checked = 0
for ph_id in multi_year_phs:
    ph = final_dataset[final_dataset['POLID'] == ph_id].sort_values('SIM_YEAR')
    for i in range(1, len(ph)):
        year_diff = ph.iloc[i]['SIM_YEAR'] - ph.iloc[i-1]['SIM_YEAR']
        age_diff = ph.iloc[i]['DRIVER_AGE'] - ph.iloc[i-1]['DRIVER_AGE']
        car_diff = ph.iloc[i]['CAR_AGE'] - ph.iloc[i-1]['CAR_AGE']
        if age_diff != year_diff:
            age_errors += 1
        if car_diff != year_diff and not (car_diff == 0 and ph.iloc[i]['CAR_AGE'] == 10):
            car_errors += 1
        prev_claim = ph.iloc[i-1]['CLAIM_OCCURRED']
        curr_ncd = ph.iloc[i]['NCD_LEVEL_PRICED']
        if prev_claim:
            total_claims_checked += 1
            if curr_ncd > 0:
                ncd_reset_failures += 1
age_ok = age_errors == 0
car_ok = car_errors == 0
ncd_ok = ncd_reset_failures == 0
print(f"\n   Age consistency (sample {len(multi_year_phs):,}): errors={age_errors} {'OK' if age_ok else 'CHECK'}")
print(f"   Car-age consistency (cap at 10 allowed): errors={car_errors} {'OK' if car_ok else 'CHECK'}")
print(f"   NCD reset on claim: failures={ncd_reset_failures}/{total_claims_checked} {'OK' if ncd_ok else 'CHECK'}")
enhanced_results.append({'Test': 'Longitudinal: Age Consistency', 'Statistic': f"{age_errors} errors", 'P-value': '-', 'Interpretation': 'DRIVER_AGE +1 per year', 'Pass': age_ok})
enhanced_results.append({'Test': 'Longitudinal: Car-Age Consistency', 'Statistic': f"{car_errors} errors", 'P-value': '-', 'Interpretation': 'CAR_AGE +1/yr (cap 10)', 'Pass': car_ok})
enhanced_results.append({'Test': 'NCD Reset on Claim', 'Statistic': f"{ncd_reset_failures}/{total_claims_checked} failures", 'P-value': '-', 'Interpretation': 'Claim in year N -> NCD 0 next year', 'Pass': ncd_ok})

# 7b. TPFT claim frequency between TPO and Comprehensive
comp_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'Comprehensive']['CLAIM_OCCURRED'].mean()
tpo_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPO']['CLAIM_OCCURRED'].mean()
tpft_freq_c = final_dataset[final_dataset['COVERAGE_TYPE'] == 'TPFT']['CLAIM_OCCURRED'].mean()
enhanced_results.append({'Test': 'TPFT Claim Frequency between TPO & Comp', 'Statistic': f"{tpo_freq_c:.1%} < {tpft_freq_c:.1%} < {comp_freq_c:.1%}", 'P-value': '-', 'Interpretation': 'TPFT covers TP + fire/theft only', 'Pass': tpo_freq_c < tpft_freq_c < comp_freq_c})

# 7. Gen Z vs Non-Gen Z
print("\n7. GEN Z vs NON-GEN Z COMPARISON")
print("-"*60)
gen_z = final_dataset[final_dataset['DRIVER_AGE'] <= 27]
non_gen_z = final_dataset[final_dataset['DRIVER_AGE'] > 27]
gen_z_pct = len(gen_z) / len(final_dataset) * 100
gz_claim = (gen_z['CLAIM_COUNT'] > 0).mean()
nz_claim = (non_gen_z['CLAIM_COUNT'] > 0).mean()
gz_prem = gen_z['FINAL_PREMIUM_SST'].mean()
nz_prem = non_gen_z['FINAL_PREMIUM_SST'].mean()
print(f"   Gen Z share: {gen_z_pct:.1f}%")
print(f"   Claim rate: Gen Z={gz_claim*100:.1f}% vs Non={nz_claim*100:.1f}%")
print(f"   Avg premium: Gen Z=RM{gz_prem:,.0f} vs Non=RM{nz_prem:,.0f}")
enhanced_results.append({'Test': 'Gen Z Share', 'Statistic': f"{gen_z_pct:.1f}%", 'P-value': '-', 'Interpretation': 'Target 25-40%', 'Pass': 25 <= gen_z_pct <= 40})
enhanced_results.append({'Test': 'Gen Z Higher Claim Rate', 'Statistic': f"{gz_claim*100:.1f}% vs {nz_claim*100:.1f}%", 'P-value': '-', 'Interpretation': 'Young drivers claim more', 'Pass': gz_claim > nz_claim})

# 8. Claim Count Poisson Fit (Var/Mean)
print("\n8. CLAIM COUNT DISTRIBUTION")
print("-"*60)
claim_counts = final_dataset['CLAIM_COUNT']
mean_claims = claim_counts.mean()
var_mean = claim_counts.var() / mean_claims
print(f"   Mean: {mean_claims:.4f}, Var/Mean: {var_mean:.3f} (1.0 = perfect Poisson)")
enhanced_results.append({'Test': 'Claim Count Poisson Fit (Var/Mean)', 'Statistic': f"{var_mean:.3f}", 'P-value': '-', 'Interpretation': 'Close to 1.0 (heterogeneity inflates)', 'Pass': 0.7 <= var_mean <= 1.6})

# 9. Log-Premium Shape
print("\n9. PREMIUM DISTRIBUTION SHAPE")
print("-"*60)
skewness = log_premiums.skew()
kurtosis = log_premiums.kurtosis()
dagostino_stat, dagostino_p = normaltest(log_premiums.sample(min(5000, len(log_premiums)), random_state=42))
print(f"   Log-premium skewness: {skewness:.3f} (target |skew| < 1.5)")
print(f"   Log-premium kurtosis: {kurtosis:.3f}")
print(f"   D'Agostino-Pearson: stat={dagostino_stat:.2f}, p={dagostino_p:.4e}")
enhanced_results.append({'Test': 'Log-Premium Skewness', 'Statistic': f"{skewness:.3f}", 'P-value': '-', 'Interpretation': '|skew| < 1.5 (tariff-fixed TPO flat premium widens left mass)', 'Pass': abs(skewness) < 1.5})

# Summary
print("\n" + "="*70)
print("ENHANCED VALIDATION SUMMARY")
print("="*70)
enhanced_df = pd.DataFrame(enhanced_results)
print("\n" + enhanced_df.to_string(index=False))
pass_count = enhanced_df['Pass'].sum()
total_count = len(enhanced_df)
print(f"\nResult: {pass_count}/{total_count} tests passed")
if pass_count == total_count:
    print("ALL ENHANCED VALIDATION TESTS PASSED")
else:
    print(f"{total_count - pass_count} test(s) failed - review above")
print("="*70)


DATASET VALIDATION


  Region premium ratio (East/Peninsular): 0.70x (report only)



                                     Test                      Value                                             Expected  Pass
Premium vs Sum Insured Correlation (Comp)                      0.794                                                > 0.5  True
             Young Driver Premium Loading                      1.16x > 1.05x (driver loading partly offset by newer cars)  True
                  Overall Claim Frequency                     14.35%                                               10-20%  True
            NCD Progression (2026 < 2030)             24.8% to 34.7%                                           Increasing  True
                       Overall Loss Ratio                     69.36%                                               50-80%  True
              Comprehensive Premium > TPO                     13.53x                                               > 1.8x  True
          TPFT Premium between TPO & Comp  RM130 < RM1,273 < RM1,761                           TPO < TP


Correlation heatmap saved: images/correlation_heatmap.png

ENHANCED STATISTICAL VALIDATION



1. PREMIUM DISTRIBUTION FIT TEST (Log-Normal)
   KS Statistic: 0.1491, p-value: 0.0000e+00
   (N=3,263,823; threshold 0.15 given coverage/NCD mixture)

2. CLAIM SEVERITY DISTRIBUTION FIT (Gamma)
------------------------------------------------------------


...

   Unique policies: 1,050,000
   Policies with 1 year: 356,103
   Policies with 2+ years: 693,897
   Policies with all 5 years: 68,995



   Age consistency (sample 1,000): errors=0 OK
   Car-age consistency (cap at 10 allowed): errors=0 OK
   NCD reset on claim: failures=0/366 OK



7. GEN Z vs NON-GEN Z COMPARISON
------------------------------------------------------------


   Gen Z share: 30.9%
   Claim rate: Gen Z=17.4% vs Non=13.0%
   Avg premium: Gen Z=RM1,529 vs Non=RM1,339

8. CLAIM COUNT DISTRIBUTION
------------------------------------------------------------
   Mean: 0.1568, Var/Mean: 1.025 (1.0 = perfect Poisson)

9. PREMIUM DISTRIBUTION SHAPE
------------------------------------------------------------


   Log-premium skewness: -1.072 (target |skew| < 1.5)
   Log-premium kurtosis: 0.501
   D'Agostino-Pearson: stat=744.08, p=2.6624e-162

ENHANCED VALIDATION SUMMARY

                                   Test             Statistic     P-value                                                Interpretation  Pass
       Premium Log-Normal Fit (KS test)             KS=0.1491  0.0000e+00                                        Mixture (TPO+Comp+NCD)  True
     Severity Gamma Fit (Comprehensive) shape=0.55, KS=0.0584  0.0000e+00                                 Per-peril mixture, shape=0.55  True
              Severity Gamma Fit (TPFT) shape=0.55, KS=0.0333  4.5423e-70                                 Per-peril mixture, shape=0.55  True
               Severity Gamma Fit (TPO) shape=0.42, KS=0.0676 8.7701e-166                                 Per-peril mixture, shape=0.42  True
         NCD: 55% tier grows over years         8.9% to 29.4%           -                          Loyal claim-free policies 

In [24]:
# ============================================================================
# OVERTHINKER-STYLE EDA (adapted from reference Section 7)
# ============================================================================
from scipy.stats import gamma as gamma_dist

print("="*70)
print("GENERATING ACTUARIAL EDA REPORTS")
print("="*70)

# 1. Actuarial Premium Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pivot_agencd = final_dataset.groupby(['age_band', 'NCD_TIER'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
sns.heatmap(pivot_agencd, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[0].set_title('Median Premium: Age Band vs NCD Tier')
axes[0].set_xlabel('NCD Tier')
axes[0].set_ylabel('Driver Age Band')

pivot_covloc = final_dataset.groupby(['COVERAGE_TYPE', 'REGION'], observed=True)['FINAL_PREMIUM_SST'].median().unstack()
pivot_covloc.columns = [c.replace(' Malaysia (Sabah, Sawarak & Labuan)', '').replace(' Malaysia', '') for c in pivot_covloc.columns]
pivot_covloc = pivot_covloc.reindex(['Comprehensive', 'TPFT', 'TPO'])
sns.heatmap(pivot_covloc, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            cbar_kws={'label': 'Median Premium (RM)'})
axes[1].set_title('Median Premium: Coverage vs Region')
plt.tight_layout()
plt.savefig('images/eda_premium_heatmaps.png', dpi=150, bbox_inches='tight')
plt.close()

# 2. Risk Profile: Frequency & Loss Ratio by Age Band
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
freq_by_age = final_dataset.groupby('age_band', observed=True)['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
sns.barplot(x=freq_by_age.index, y=freq_by_age.values, ax=axes[0], palette='viridis')
axes[0].axhline((final_dataset['CLAIM_COUNT'] > 0).mean() * 100, ls='--', color='red', label='Portfolio Average')
axes[0].set_title('Claim Frequency by Age Band (U-Shaped Risk)')
axes[0].set_ylabel('Claim Frequency (%)')
axes[0].legend()

lr_by_age = final_dataset.groupby('age_band', observed=True).apply(
    lambda x: (x['CLAIM_AMOUNT'].sum() / x['FINAL_PREMIUM_SST'].sum()) * 100
)
sns.barplot(x=lr_by_age.index, y=lr_by_age.values, ax=axes[1], palette='magma')
axes[1].axhline((final_dataset['CLAIM_AMOUNT'].sum() / final_dataset['FINAL_PREMIUM_SST'].sum()) * 100,
                ls='--', color='red', label='Portfolio Average')
axes[1].set_title('Loss Ratio by Age Band')
axes[1].set_ylabel('Loss Ratio (%)')
axes[1].legend()
plt.tight_layout()
plt.savefig('images/eda_risk_profile.png', dpi=150, bbox_inches='tight')
plt.close()

# 3. Severity Tails (Log-Scale)
plt.figure(figsize=(10, 6))
claimants = final_dataset[final_dataset['CLAIM_AMOUNT'] > 0]
sns.histplot(data=claimants, x=np.log1p(claimants['CLAIM_AMOUNT']),
             hue='COVERAGE_TYPE', hue_order=['Comprehensive', 'TPFT', 'TPO'],
             bins=50, kde=True, palette='Set2', alpha=0.6, element='step')
plt.title('Log-Scale Claim Severity Distribution (Fat Tails)')
plt.xlabel('log(1 + Claim Amount)')
plt.ylabel('Count of Claims')
plt.savefig('images/eda_severity_tails.png', dpi=150, bbox_inches='tight')
plt.close()

# 4. Severity Distributions by Coverage (Gamma overlay)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, cov in enumerate(['Comprehensive', 'TPFT', 'TPO']):
    claims_subset = final_dataset[(final_dataset['COVERAGE_TYPE'] == cov) & (final_dataset['CLAIM_AMOUNT'] > 0)]['CLAIM_AMOUNT']
    if len(claims_subset) > 10:
        claims_subset.hist(bins=50, ax=axes[i], color=['skyblue', 'coral', 'seagreen'][i],
                           edgecolor='black', alpha=0.7, density=True)
        shape_f, loc_f, scale_f = gamma_dist.fit(claims_subset, floc=0)
        x = np.linspace(0, claims_subset.quantile(0.99), 200)
        axes[i].plot(x, gamma_dist.pdf(x, shape_f, loc_f, scale_f), 'r-', lw=2,
                     label=f'Gamma fit (k={shape_f:.1f})')
        axes[i].set_title(f'{cov} Severity Distribution', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('Claim Amount (RM)')
        axes[i].legend()
        axes[i].axvline(claims_subset.mean(), color='black', linestyle='--', alpha=0.5, label='Mean')
        axes[i].axvline(claims_subset.quantile(0.95), color='red', linestyle=':', alpha=0.5, label='P95')
plt.tight_layout()
plt.savefig('images/severity_distributions.png', dpi=150, bbox_inches='tight')
plt.close()

# 5. Longitudinal Trends
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
year_stats = final_dataset.groupby('SIM_YEAR').agg({
    'FINAL_PREMIUM_SST': 'mean',
    'NCD_LEVEL_PRICED': 'mean',
    'CLAIM_COUNT': lambda x: (x > 0).mean()
}).reset_index()
axes[0].plot(year_stats['SIM_YEAR'], year_stats['FINAL_PREMIUM_SST'], 'b-o', linewidth=2, markersize=8)
axes[0].set_title('Average Premium by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('SIM_YEAR')
axes[0].set_ylabel('Avg Premium (RM)')
axes[0].grid(True, alpha=0.3)
ax2 = axes[0].twinx()
ax2.plot(year_stats['SIM_YEAR'], year_stats['NCD_LEVEL_PRICED']*100, 'r--s', linewidth=2, markersize=6, alpha=0.7)
ax2.set_ylabel('Avg NCD %', color='red')

for label, age_min, age_max, color in [('Gen Z (18-27)', 18, 27, 'red'),
                                       ('Prime (28-50)', 28, 50, 'blue'),
                                       ('Senior (51+)', 51, 100, 'green')]:
    subset = final_dataset[(final_dataset['DRIVER_AGE'] >= age_min) & (final_dataset['DRIVER_AGE'] <= age_max)]
    rates = subset.groupby('SIM_YEAR')['CLAIM_COUNT'].apply(lambda x: (x > 0).mean() * 100)
    axes[1].plot(rates.index, rates.values, '-o', label=label, color=color, linewidth=2)
axes[1].set_title('Claim Rate by Year & Age Group', fontsize=14, fontweight='bold')
axes[1].set_xlabel('SIM_YEAR')
axes[1].set_ylabel('Claim Rate (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('images/longitudinal_trends.png', dpi=150, bbox_inches='tight')
plt.close()

print("EDA visualisations generated and saved:")
print("  images/correlation_heatmap.png, images/ncd_validation.png")
print("  images/eda_premium_heatmaps.png, images/eda_risk_profile.png")
print("  images/eda_severity_tails.png, images/severity_distributions.png")
print("  images/longitudinal_trends.png")


GENERATING ACTUARIAL EDA REPORTS


C:\Users\Admin\AppData\Local\Temp\ipykernel_9380\2686387801.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=freq_by_age.index, y=freq_by_age.values, ax=axes[0], palette='viridis')


C:\Users\Admin\AppData\Local\Temp\ipykernel_9380\2686387801.py:41: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=lr_by_age.index, y=lr_by_age.values, ax=axes[1], palette='magma')


EDA visualisations generated and saved:
  images/correlation_heatmap.png, images/ncd_validation.png
  images/eda_premium_heatmaps.png, images/eda_risk_profile.png
  images/eda_severity_tails.png, images/severity_distributions.png
  images/longitudinal_trends.png


In [25]:
df.head()

,COVERAGE_TYPE,VEHICLE_TYPE,REGION,SUM_ASSURED,ENGINE_CAPACITY,DRIVER_AGE_CAT,DRIVER_AGE,DRIVER_GENDER,MARITAL_STATUS,CAR_AGE,FLOOD_RISK,THEFT_RISK,NCD_YEARS,NCD_LEVEL,COHORT_YEAR,POLID,BASIC_PREMIUM,TOTAL_LOADING,FINAL_PREMIUM_SST,CLAIM_LAMBDA
0,TPFT,EV,Peninsular Malaysia,48000.0,"1,651 - 2,200 cc / EV 101 - 125 kW",Millennial,34,Male,Married,4,True,False,0,0.0,2026,INIT2026-000001,1170.83,1.176,1635.75,0.117558
1,Comprehensive,EV,Peninsular Malaysia,113000.0,"2,201 - 3,050 cc / EV 126 - 150 kW",Millennial,44,Female,Married,2,False,False,0,0.0,2026,INIT2026-000002,3284.60,1.113,3948.22,0.151072
2,TPO,ICE,"East Malaysia (Sabah, Sawarak & Labuan)",80000.0,"2,201 - 3,050 cc / EV 126 - 150 kW",Boomers,53,Female,Married,4,False,False,0,0.0,2026,INIT2026-000003,93.60,1.120,113.22,0.068666
3,TPFT,ICE,Peninsular Malaysia,63000.0,"1,401 to 1,650 cc / EV 71 - 100 kW",Millennial,34,Female,Married,2,False,False,2,0.3,2026,INIT2026-000004,1438.13,1.113,1210.08,0.078017
4,Comprehensive,ICE,Peninsular Malaysia,45000.0,"2,201 - 3,050 cc / EV 126 - 150 kW",Gen-Z,18,Male,Single,2,False,False,2,0.3,2026,INIT2026-000005,1516.60,1.272,1458.41,0.203926


In [26]:
# ============================================================================
# EXPORT SIMULATION DATA TO CSV (data/)
# Reproducible: deterministic (seed 42), regenerated on every execution
# ============================================================================
import os, shutil

os.makedirs('data', exist_ok=True)

cohort_results.to_csv('data/simulation_cohort_results.csv', index=False)
df.to_csv('data/initial_book.csv', index=False)

params = pd.DataFrame([
    {'PARAMETER': 'NCD_TABLE',           'VALUE': repr(NCD_TABLE)},
    {'PARAMETER': 'DRIVER_AGE_LOADING',  'VALUE': repr(DRIVER_AGE_LOADING)},
    {'PARAMETER': 'CAR_AGE_LOADING',     'VALUE': '1 + 0.03 * min(car_age, 10)'},
    {'PARAMETER': 'SST_RATE',            'VALUE': repr(SST_RATE)},
    {'PARAMETER': 'RISK_LOADING_FLOOD',  'VALUE': '1.10'},
    {'PARAMETER': 'RISK_LOADING_THEFT',  'VALUE': '1.10'},
    {'PARAMETER': 'PERIL_DIST',          'VALUE': repr(PERIL_DIST)},
    {'PARAMETER': 'PERIL_BASE',          'VALUE': repr(PERIL_BASE)},
    {'PARAMETER': 'NCD_ENTRY_YEARS',     'VALUE': repr(NCD_ENTRY_YEARS)},
    {'PARAMETER': 'NCD_ENTRY_WEIGHTS',   'VALUE': repr(NCD_ENTRY_WEIGHTS)},
    {'PARAMETER': 'CLAIM_LAMBDA_BASE',   'VALUE': '-2.00 (log rate)'},
    {'PARAMETER': 'TPO_FREQ_MULTIPLIER', 'VALUE': '0.45'},
    {'PARAMETER': 'TPFT_FREQ_MULTIPLIER', 'VALUE': '0.60'},
    {'PARAMETER': 'COHORT_YEAR',         'VALUE': repr(COHORT_YEAR)},
    {'PARAMETER': 'NUM_DATASET',         'VALUE': repr(num_dataset)},
    {'PARAMETER': 'SIM_YEARS',           'VALUE': repr(sorted(cohort_results['SIM_YEAR'].unique()))},
    {'PARAMETER': 'SEED',                'VALUE': '42'},
    {'PARAMETER': 'AGING_CAP_CAR_AGE',   'VALUE': '10'},
])
params.to_csv('data/model_parameters.csv', index=False)

print('Exports written to data/:')
for f in sorted(os.listdir('data')):
    p = os.path.join('data', f)
    print(f'  {f:38s} {os.path.getsize(p):>12,} bytes')
print(f'cohort_results: {len(cohort_results):,} rows x {cohort_results.shape[1]} cols')
print(f'initial book  : {len(df):,} rows x {df.shape[1]} cols')

Exports written to data/:
  initial_book.csv                         19,276,660 bytes
  model_parameters.csv                          1,285 bytes
  simulation_cohort_results.csv           661,412,982 bytes
cohort_results: 3,263,823 rows x 27 cols
initial book  : 100,000 rows x 20 cols
